## CELL 00 — Binary Left-vs-Right Motor-Imagery EEG Pipeline

This notebook is the binary version of the supplied S³ Spectral-Spatial Domain Adaptation pipeline.

**Task:** subject-independent classification of **Left-Hand MI vs Right-Hand MI** using PhysioNet EEGMMIDB / EEG Motor Movement-Imagery runs **4, 8, 12** only.

**Architecture:** per-trial channel Z-score → learnable Sinc filter bank → dynamic GNN → bidirectional GRU (the supplied `SimplifiedBiMamba` implementation) → SE attention → classification + GRL domain alignment + supervised contrastive learning → AdaBN target-statistics adaptation.

**Important:** this code uses a bidirectional GRU, not a true Mamba state-space model. The latent diffusion module described in the reference manuscript is not implemented.


## CELL 01 — Imports and environment

This cell imports the numerical, EEG, deep-learning, and evaluation libraries.

The important packages are:
- **MNE** for reading PhysioNet EEGMMIDB EDF files, annotations, and epoching.
- **PyTorch** for the model and training.
- **scikit-learn** for accuracy, Cohen's kappa, classification metrics, ROC/AUC, and t-SNE.
- **pandas / NumPy / Matplotlib** for experiment logging and figures.

In [1]:
# ============================================================
# CELL 01 — Imports and environment
# ============================================================

from pathlib import Path
import os, math, json, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    confusion_matrix, classification_report, roc_curve, auc,
    roc_auc_score
)
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

print("Imports loaded successfully.")


Imports loaded successfully.


## CELL 02 — Configuration and reproducibility

This cell defines the experimental settings.

The source configuration uses:
- 109 possible subjects
- runs 4, 6, 8, 10, 12, and 14
- 250 Hz sampling
- 4-second trials
- 22 EEG channels
- 4 classes
- batch size 64
- 100 training epochs
- AdamW with learning rate `1e-3`
- cosine annealing
- label smoothing, domain loss, and supervised contrastive loss

A fixed random seed of 42 is used. The code automatically selects CUDA, then Apple MPS, then CPU. fileciteturn1file0L76-L137

In [35]:
# ============================================================
# CELL 02 — BINARY LEFT-vs-RIGHT OPTIMIZED CONFIGURATION
# ============================================================

SEED = 42

# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------

DATA_DIR = "./eegmmidb"

# Only Left/Right motor imagery runs
RUNS = [4, 8, 12]

TMIN = 0.0
TMAX = 4.0
FS = 250.0

N_CHANNELS = 22

# ------------------------------------------------------------
# BINARY CLASSES
# ------------------------------------------------------------

N_CLASSES = 2

CLASS_NAMES = [
    "Left Hand MI",
    "Right Hand MI"
]

# ------------------------------------------------------------
# EXACT TEST SUBJECTS REQUESTED
# ------------------------------------------------------------

TEST_SUBJECTS = [
    4,
    15,
    23,
    29,
    31,
    42,
    55,
    71,
    82,
    95
]

NUM_TEST_FOLDS = len(TEST_SUBJECTS)

# ------------------------------------------------------------
# SOURCE TRAINING SUBJECTS
# ------------------------------------------------------------

TOTAL_SUBJECTS = 109

# All available source subjects except the target
USE_ALL_SOURCE_SUBJECTS = True

# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

BATCH_SIZE = 64

NUM_EPOCHS = 100

LR = 3e-4

MIN_LR = 1e-6

WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.03

GRAD_CLIP = 1.0

# ------------------------------------------------------------
# DOMAIN ADVERSARIAL LEARNING
# ------------------------------------------------------------
#
# Reduced so binary classification dominates the objective.
# ------------------------------------------------------------

DOMAIN_WEIGHT = 0.10

# ------------------------------------------------------------
# SUPERVISED CONTRASTIVE LEARNING
# ------------------------------------------------------------

SUPCON_WEIGHT = 0.10

SUPCON_TEMP = 0.10

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

NUM_FILTERS = 10

SINC_KERNEL = 81

SPATIAL_DIM = 64

# ------------------------------------------------------------
# TEMPORAL COMPRESSION
# ------------------------------------------------------------

TEMPORAL_DOWNSAMPLE = 4

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

RESULTS_DIR = Path(
    "./results_binary_exact10_optimized"
)

FIG_DIR = (
    RESULTS_DIR /
    "figures"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DOMAIN_CLASSES = TOTAL_SUBJECTS

print("=" * 80)
print("BINARY LEFT-vs-RIGHT OPTIMIZED CONFIGURATION")
print("=" * 80)

print("Classes:")
print("  0 =", CLASS_NAMES[0])
print("  1 =", CLASS_NAMES[1])

print("\nRuns:", RUNS)

print(
    "\nExact test subjects:",
    [
        f"S{s:03d}"
        for s in TEST_SUBJECTS
    ]
)

print("\nTraining configuration:")
print("  Batch size       :", BATCH_SIZE)
print("  Epochs           :", NUM_EPOCHS)
print("  Learning rate    :", LR)
print("  Weight decay     :", WEIGHT_DECAY)
print("  Domain weight    :", DOMAIN_WEIGHT)
print("  SupCon weight    :", SUPCON_WEIGHT)
print("  Temporal factor  :", TEMPORAL_DOWNSAMPLE)

print("\n[OK] Configuration loaded.")

BINARY LEFT-vs-RIGHT OPTIMIZED CONFIGURATION
Classes:
  0 = Left Hand MI
  1 = Right Hand MI

Runs: [4, 8, 12]

Exact test subjects: ['S004', 'S015', 'S023', 'S029', 'S031', 'S042', 'S055', 'S071', 'S082', 'S095']

Training configuration:
  Batch size       : 64
  Epochs           : 100
  Learning rate    : 0.0003
  Weight decay     : 0.0001
  Domain weight    : 0.1
  SupCon weight    : 0.1
  Temporal factor  : 4

[OK] Configuration loaded.


## CELL 03 — Dataset loader

`EEGMMIDB_Dataset` scans subject folders (`S001`, `S002`, …), loads the selected EDF runs, keeps the first 22 channels, resamples to 250 Hz, reads annotation events, and creates 4-second epochs.

The run-dependent label mapping is:

| Runs | T1 | T2 |
|---|---|---|
| 4, 8, 12 | Left Fist = 0 | Right Fist = 1 |
| 6, 10, 14 | Both Fists = 2 | Both Feet = 3 |

The dataset also stores a **subject ID** for the domain-adaptation loss and the run ID for traceability. Each trial is then normalized independently, channel by channel, using its own temporal mean and standard deviation. fileciteturn1file0L143-L240

In [36]:
# ============================================================
# CELL 03 — Binary EEGMMIDB dataset loader
# ============================================================

class EEGMMIDB_Dataset(Dataset):
    """
    Binary EEGMMIDB loader for Left-Hand MI vs Right-Hand MI.

    Only runs 4, 8 and 12 are used. In these runs:
        T1 -> Left Hand MI
        T2 -> Right Hand MI

    MNE's events_from_annotations() returns the ACTUAL numeric
    event IDs. Those IDs must be used for MNE Epochs. We then
    convert T1/T2 into our ML labels 0/1.
    """

    def __init__(self, data_dir, subjects, runs=RUNS, tmin=TMIN, tmax=TMAX):
        self.data_dir = str(data_dir)
        self.subjects = list(subjects)
        self.runs = list(runs)
        self.tmin = tmin
        self.tmax = tmax

        self.epochs = []
        self.labels = []
        self.subject_ids = []
        self.run_ids = []

        self.load_data()

    @staticmethod
    def _find_event_code(event_id_dict, target_name):
        target_name = str(target_name).strip().upper()

        for description, code in event_id_dict.items():
            desc = str(description).strip().upper()

            if desc == target_name:
                return int(code)

            if desc.startswith(target_name):
                remainder = desc[len(target_name):]
                if (
                    remainder == ""
                    or remainder.startswith("/")
                    or remainder.startswith("-")
                    or remainder.startswith("_")
                    or remainder.isspace()
                ):
                    return int(code)

        return None

    def load_data(self):
        total_loaded = 0

        for sub in self.subjects:
            sub_folder = f"S{sub:03d}"
            sub_path = os.path.join(self.data_dir, sub_folder)

            if not os.path.isdir(sub_path):
                continue

            for run in self.runs:
                edf_file = os.path.join(
                    sub_path,
                    f"{sub_folder}R{run:02d}.edf"
                )

                if not os.path.isfile(edf_file):
                    continue

                try:
                    raw = mne.io.read_raw_edf(
                        edf_file,
                        preload=True,
                        verbose=False
                    )

                    if len(raw.ch_names) < N_CHANNELS:
                        print(
                            f"[WARN] {sub_folder} R{run:02d}: "
                            f"only {len(raw.ch_names)} channels found; "
                            f"expected {N_CHANNELS}."
                        )
                        continue

                    # Keep first 22 channels, matching the supplied pipeline.
                    raw.pick(raw.ch_names[:N_CHANNELS])
                    raw.resample(FS, npad="auto")

                    events, event_id_dict = mne.events_from_annotations(
                        raw, verbose=False
                    )

                    t1_code = self._find_event_code(
                        event_id_dict, "T1"
                    )
                    t2_code = self._find_event_code(
                        event_id_dict, "T2"
                    )

                    if t1_code is None or t2_code is None:
                        print(
                            f"[WARN] {sub_folder} R{run:02d}: "
                            f"T1/T2 not found. Available annotations: "
                            f"{list(event_id_dict.keys())}"
                        )
                        continue

                    # IMPORTANT: these are MNE event codes, not class labels.
                    event_selection = {
                        "T1": t1_code,
                        "T2": t2_code
                    }

                    ep = mne.Epochs(
                        raw,
                        events,
                        event_id=event_selection,
                        tmin=self.tmin,
                        tmax=self.tmax - 1.0 / FS,
                        baseline=None,
                        preload=True,
                        reject_by_annotation=True,
                        verbose=False
                    )

                    if len(ep) == 0:
                        print(
                            f"[WARN] {sub_folder} R{run:02d}: "
                            f"T1/T2 found but zero epochs created."
                        )
                        continue

                    data = ep.get_data()
                    actual_codes = ep.events[:, -1]

                    for i in range(len(data)):
                        actual_code = int(actual_codes[i])

                        # Binary ML mapping:
                        # T1 = 0 = Left Hand MI
                        # T2 = 1 = Right Hand MI
                        if actual_code == t1_code:
                            class_label = 0
                        elif actual_code == t2_code:
                            class_label = 1
                        else:
                            continue

                        self.epochs.append(
                            data[i].astype(np.float32)
                        )
                        self.labels.append(int(class_label))
                        self.subject_ids.append(int(sub - 1))
                        self.run_ids.append(int(run))
                        total_loaded += 1

                    print(
                        f"[OK] {sub_folder} R{run:02d} | "
                        f"T1={t1_code}->Left | "
                        f"T2={t2_code}->Right | "
                        f"epochs={len(ep)}"
                    )

                except Exception as e:
                    print(
                        f"[WARN] {sub_folder} R{run:02d}: "
                        f"{type(e).__name__}: {e}"
                    )

        print(
            f"\nDataset loading complete: "
            f"{total_loaded} binary trials"
        )

    def __len__(self):
        return len(self.epochs)

    def __getitem__(self, idx):
        x = torch.tensor(self.epochs[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        s = torch.tensor(self.subject_ids[idx], dtype=torch.long)

        # Per-trial, per-channel Z-score normalization.
        mean = x.mean(dim=1, keepdim=True)
        std = x.std(dim=1, keepdim=True)
        x = (x - mean) / (std + 1e-6)

        return x, y, s


def discover_available_subjects(data_dir, max_subjects=TOTAL_SUBJECTS):
    available = []
    for s in range(1, max_subjects + 1):
        if os.path.isdir(Path(data_dir) / f"S{s:03d}"):
            available.append(s)
    return available


# ------------------------------------------------------------
# Binary dataset sanity check
# ------------------------------------------------------------
available_subjects = discover_available_subjects(
    DATA_DIR, TOTAL_SUBJECTS
)

print("Available subject folders:", len(available_subjects))

if available_subjects:
    test_subject = available_subjects[0]
    print(f"Testing binary loader on S{test_subject:03d}")

    test_dataset = EEGMMIDB_Dataset(
        DATA_DIR,
        subjects=[test_subject]
    )

    print("Number of trials:", len(test_dataset))

    if len(test_dataset) > 0:
        labels = np.array(test_dataset.labels)
        unique, counts = np.unique(labels, return_counts=True)

        print("Class distribution:")
        for c, n in zip(unique, counts):
            print(f"  {c} ({CLASS_NAMES[c]}): {n}")

        x0, y0, s0 = test_dataset[0]
        print("Example tensor shape:", tuple(x0.shape))
        print("Example label:", int(y0))
        print("Example subject ID:", int(s0))
        print("[OK] Binary dataset loader is ready.")


Available subject folders: 109
Testing binary loader on S001
[OK] S001 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S001 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S001 R12 | T1=2->Left | T2=3->Right | epochs=15

Dataset loading complete: 45 binary trials
Number of trials: 45
Class distribution:
  0 (Left Hand MI): 23
  1 (Right Hand MI): 22
Example tensor shape: (22, 1000)
Example label: 1
Example subject ID: 0
[OK] Binary dataset loader is ready.


## CELL 04 — Gradient Reversal and learnable Sinc filter bank

This part starts the feature extractor.

### Gradient Reversal Layer
During forward propagation the GRL behaves like an identity operation. During backpropagation it multiplies the gradient by `-lambda`. Therefore:
- the domain classifier learns to predict the subject;
- the feature extractor receives the **opposite** gradient and is encouraged to make subjects harder to distinguish.

### Learnable Sinc filter bank
The model learns two frequency cutoffs for each of 10 filters. Each filter is constructed as a band-pass Sinc kernel and applied independently to every EEG channel.

Input shape:
`(B, 22, 1000)`

Output shape:
`(B, 10, 22, 1000)`

So the raw temporal signal becomes a learned set of 10 spectral representations. fileciteturn1file0L254-L309

In [37]:
# ============================================================
# CELL 04 — GRL and SincFilterBank
# ============================================================

# -----------------------------
# 2. ARCHITECTURE
# -----------------------------

class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_grl):
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_grl, None

def grl(x, lambda_grl=1.0):
    return GradientReversalLayer.apply(x, lambda_grl)





## CELL 05 — Dynamic graph neural network

`DGNN` builds an adaptive channel graph from the spectral features.

For each trial it:
1. averages over time to obtain channel descriptors;
2. projects the descriptors into query and key spaces;
3. computes pairwise channel similarity;
4. applies softmax to obtain a dynamic adjacency matrix;
5. adds self-connections;
6. degree-normalizes the adjacency matrix;
7. mixes information across EEG channels;
8. projects the resulting 22-node representation into a 64-dimensional spatial feature space.

The returned tensor has shape:
`(B, 10 bands, 64 spatial features, T)`. fileciteturn1file0L312-L343

In [38]:
# ============================================================
# CELL 05 — Dynamic GNN
# ============================================================

class SincFilterBank(nn.Module):
    def __init__(self, in_channels=22, num_filters=10, kernel_size=81, sample_rate=250):
        super().__init__()
        self.num_filters = num_filters
        self.kernel_size = kernel_size
        self.sample_rate = sample_rate

        # Same initialization family as the supplied code.
        self.f1 = nn.Parameter(torch.rand(num_filters) * 10 + 5)
        self.f2 = nn.Parameter(torch.rand(num_filters) * 20 + 15)

    def forward(self, x):
        B, C, T = x.shape

        n = torch.arange(
            -(self.kernel_size // 2),
            (self.kernel_size // 2) + 1,
            device=x.device,
            dtype=x.dtype
        )

        filters = []
        for i in range(self.num_filters):
            # Sorted positive cutoffs make the implementation numerically safer
            # without changing the conceptual learnable-Sinc design.
            lo = torch.minimum(self.f1[i], self.f2[i] - 1e-3).clamp(0.5, 70.0)
            hi = torch.maximum(self.f2[i], self.f1[i] + 1e-3).clamp(1.0, 95.0)

            f1_scaled = lo / self.sample_rate
            f2_scaled = hi / self.sample_rate

            w = (
                2 * f2_scaled * torch.sinc(2 * f2_scaled * n)
                - 2 * f1_scaled * torch.sinc(2 * f1_scaled * n)
            )
            filters.append(w.unsqueeze(0).unsqueeze(0))

        filters = torch.cat(filters, dim=0)
        x_reshaped = x.reshape(B * C, 1, T)
        out = F.conv1d(x_reshaped, filters, padding="same")

        return out.reshape(B, C, self.num_filters, T).permute(0, 2, 1, 3)





## CELL 06 — Temporal block and SE attention

The class named `SimplifiedBiMamba` is a bidirectional GRU.

After averaging the 10 spectral bands, the tensor is:
`(B, 64, T)`.

The GRU reads the time axis in both directions and returns another `(B, 64, T)` representation.

`SEAttention` then:
1. averages each feature over time;
2. uses a small bottleneck MLP and sigmoid to generate feature weights;
3. reweights the temporal features;
4. averages over time to produce the final 64-dimensional embedding.

This 64-D representation is the shared feature used by the task, domain, and contrastive objectives. fileciteturn1file0L346-L381

In [39]:
# ============================================================
# CELL 06 — OPTIMIZED TEMPORAL ENCODER + SE ATTENTION
# ============================================================

class TemporalDownsample(nn.Module):
    """
    Lightweight temporal feature extractor.

    Input:
        [B, C, T]

    Output:
        [B, C, T/4] approximately

    This reduces the sequence length before the BiGRU.
    """

    def __init__(self, channels=64):
        super().__init__()

        self.net = nn.Sequential(

            # ------------------------------------------------
            # First temporal convolution
            # ------------------------------------------------
            nn.Conv1d(
                channels,
                channels,
                kernel_size=7,
                stride=2,
                padding=3,
                groups=channels,
                bias=False
            ),

            nn.BatchNorm1d(channels),
            nn.ELU(inplace=True),

            # ------------------------------------------------
            # Second temporal convolution
            # ------------------------------------------------
            nn.Conv1d(
                channels,
                channels,
                kernel_size=7,
                stride=2,
                padding=3,
                groups=channels,
                bias=False
            ),

            nn.BatchNorm1d(channels),
            nn.ELU(inplace=True),

            # ------------------------------------------------
            # Pointwise feature mixing
            # ------------------------------------------------
            nn.Conv1d(
                channels,
                channels,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm1d(channels),
            nn.ELU(inplace=True)
        )

    def forward(self, x):

        # x:
        # [B, C, T]

        return self.net(x)


# ============================================================
# BIDIRECTIONAL TEMPORAL GRU
# ============================================================

class SimplifiedBiMamba(nn.Module):
    """
    The original implementation calls this BiMamba,
    but technically it is a Bidirectional GRU.

    Input:
        [B, D, T]

    Output:
        [B, D, T]
    """

    def __init__(self, d_model=64):
        super().__init__()

        self.ssm = nn.GRU(
            input_size=d_model,
            hidden_size=d_model // 2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):

        # [B, D, T] -> [B, T, D]
        x_seq = x.transpose(1, 2)

        out, _ = self.ssm(x_seq)

        # [B, T, D] -> [B, D, T]
        return out.transpose(1, 2)


# ============================================================
# SE ATTENTION
# ============================================================

class SEAttention(nn.Module):

    def __init__(
        self,
        channel=64,
        reduction=16
    ):
        super().__init__()

        hidden_dim = max(
            1,
            channel // reduction
        )

        self.fc = nn.Sequential(

            nn.Linear(
                channel,
                hidden_dim,
                bias=False
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                hidden_dim,
                channel,
                bias=False
            ),

            nn.Sigmoid()
        )

    def forward(self, x):

        # x = [B, C, T]

        b, c, t = x.size()

        # Global temporal pooling
        y = x.mean(dim=2)

        # Channel attention
        weights = self.fc(y)

        weights = weights.view(
            b,
            c,
            1
        )

        # Re-weight channels
        x = x * weights

        # Final temporal pooling
        return x.mean(dim=2)


# ============================================================
# VERIFY
# ============================================================

print("=" * 70)
print("CELL 06 CHECK")
print("=" * 70)

test_x = torch.randn(
    2,
    SPATIAL_DIM,
    int(FS * TMAX)
).to(DEVICE)

with torch.no_grad():

    temporal_downsample = TemporalDownsample(
        SPATIAL_DIM
    ).to(DEVICE)

    temporal_gru = SimplifiedBiMamba(
        SPATIAL_DIM
    ).to(DEVICE)

    se_attention = SEAttention(
        SPATIAL_DIM
    ).to(DEVICE)

    x_down = temporal_downsample(
        test_x
    )

    x_gru = temporal_gru(
        x_down
    )

    x_se = se_attention(
        x_gru
    )

print("Input               :", tuple(test_x.shape))
print("After downsampling  :", tuple(x_down.shape))
print("After BiGRU         :", tuple(x_gru.shape))
print("After SE attention  :", tuple(x_se.shape))

print()
print("[OK] Optimized temporal module ready.")

CELL 06 CHECK
Input               : (2, 64, 1000)
After downsampling  : (2, 64, 250)
After BiGRU         : (2, 64, 250)
After SE attention  : (2, 64)

[OK] Optimized temporal module ready.


## CELL 07 — Full S³ Binary Domain Adaptation model

`S3MambaDA` connects the complete feature extractor and the three training heads:

**Input → Sinc → DGNN → band pooling → BiGRU → SE attention → 64-D embedding**

Then:
- **Classifier:** 64 → 4 class logits.
- **Domain classifier:** GRL → 64 → 32 → 109 subject logits.
- **SupCon projection:** 64 → 128 → 128, followed by L2 normalization.

The forward pass therefore returns three objects:
`class_logits, domain_logits, z_proj`. fileciteturn1file0L384-L436

In [40]:
# ============================================================
# CELL 07 — OPTIMIZED S3 BINARY MODEL
# ============================================================

class S3MambaDA(nn.Module):

    def __init__(
        self,
        num_classes=2,
        num_subjects=109
    ):
        super().__init__()

        # ----------------------------------------------------
        # Spectral decomposition
        # ----------------------------------------------------

        self.sinc_filter = SincFilterBank(
            in_channels=N_CHANNELS,
            num_filters=NUM_FILTERS,
            kernel_size=SINC_KERNEL,
            sample_rate=int(FS)
        )

        # ----------------------------------------------------
        # Spatial graph learning
        # ----------------------------------------------------

        self.dgnn = DGNN(
            num_filters=NUM_FILTERS,
            in_nodes=N_CHANNELS,
            out_nodes=SPATIAL_DIM
        )

        # ----------------------------------------------------
        # Temporal downsampling
        # ----------------------------------------------------

        self.temporal_downsample = TemporalDownsample(
            channels=SPATIAL_DIM
        )

        # ----------------------------------------------------
        # Temporal modeling
        # ----------------------------------------------------

        self.mamba = SimplifiedBiMamba(
            d_model=SPATIAL_DIM
        )

        # ----------------------------------------------------
        # SE attention
        # ----------------------------------------------------

        self.se_attention = SEAttention(
            channel=SPATIAL_DIM
        )

        # ----------------------------------------------------
        # Binary classifier
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.BatchNorm1d(
                SPATIAL_DIM
            ),

            nn.Dropout(
                p=0.20
            ),

            nn.Linear(
                SPATIAL_DIM,
                num_classes
            )
        )

        # ----------------------------------------------------
        # Domain classifier
        # ----------------------------------------------------

        self.domain_classifier = nn.Sequential(

            nn.Linear(
                SPATIAL_DIM,
                64
            ),

            nn.ReLU(inplace=True),

            nn.Dropout(
                p=0.20
            ),

            nn.Linear(
                64,
                num_subjects
            )
        )

        # ----------------------------------------------------
        # Supervised contrastive projection
        # ----------------------------------------------------

        self.supcon_proj = nn.Sequential(

            nn.Linear(
                SPATIAL_DIM,
                128
            ),

            nn.ReLU(inplace=True),

            nn.Dropout(
                p=0.10
            ),

            nn.Linear(
                128,
                128
            )
        )

    def forward(
        self,
        x,
        lambda_grl=1.0
    ):

        # ----------------------------------------------------
        # 1. Learnable spectral decomposition
        # ----------------------------------------------------

        f_out = self.sinc_filter(x)
        # [B, 10, 22, T]

        # ----------------------------------------------------
        # 2. Dynamic spatial graph
        # ----------------------------------------------------

        s_out = self.dgnn(f_out)
        # [B, 10, 64, T]

        # ----------------------------------------------------
        # 3. Average spectral representations
        # ----------------------------------------------------

        pool_out = s_out.mean(
            dim=1
        )
        # [B, 64, T]

        # ----------------------------------------------------
        # 4. Temporal downsampling
        # ----------------------------------------------------

        pool_out = self.temporal_downsample(
            pool_out
        )
        # [B, 64, ~250]

        # ----------------------------------------------------
        # 5. Temporal modeling
        # ----------------------------------------------------

        t_out = self.mamba(
            pool_out
        )
        # [B, 64, ~250]

        # ----------------------------------------------------
        # 6. SE attention
        # ----------------------------------------------------

        z = self.se_attention(
            t_out
        )
        # [B, 64]

        # ----------------------------------------------------
        # 7. Binary classification
        # ----------------------------------------------------

        class_logits = self.classifier(
            z
        )

        # ----------------------------------------------------
        # 8. Domain adversarial learning
        # ----------------------------------------------------

        z_grl = grl(
            z,
            lambda_grl
        )

        domain_logits = (
            self.domain_classifier(
                z_grl
            )
        )

        # ----------------------------------------------------
        # 9. SupCon embedding
        # ----------------------------------------------------

        z_proj = F.normalize(
            self.supcon_proj(z),
            p=2,
            dim=1
        )

        return (
            class_logits,
            domain_logits,
            z_proj
        )


print("=" * 70)
print("CELL 07 CHECK")
print("=" * 70)

model_test = S3MambaDA(
    num_classes=N_CLASSES,
    num_subjects=DOMAIN_CLASSES
).to(DEVICE)

test_input = torch.randn(
    2,
    N_CHANNELS,
    int(FS * TMAX)
).to(DEVICE)

with torch.no_grad():

    class_logits, domain_logits, z_proj = model_test(
        test_input,
        lambda_grl=0.0
    )

print("Input          :", tuple(test_input.shape))
print("Class logits   :", tuple(class_logits.shape))
print("Domain logits  :", tuple(domain_logits.shape))
print("SupCon         :", tuple(z_proj.shape))
print(
    "Parameters     :",
    f"{sum(p.numel() for p in model_test.parameters()):,}"
)

print()
print("[OK] Optimized model passed smoke test.")

del model_test
del test_input
del class_logits
del domain_logits
del z_proj

CELL 07 CHECK
Input          : (2, 22, 1000)
Class logits   : (2, 2)
Domain logits  : (2, 109)
SupCon         : (2, 128)
Parameters     : 62,751

[OK] Optimized model passed smoke test.


## CELL 08 — Losses and AdaBN adaptation

The training objective is the sum of three components:

`Total = Classification Loss + 1.0 × Domain Loss + 0.5 × SupCon Loss`

The classification loss uses cross-entropy with label smoothing. The domain loss predicts the subject identity. The supervised contrastive loss uses class labels so trials from the same class act as positives.

After training on source subjects, `apply_adabn()` collects unlabeled target trials and refreshes BatchNorm running statistics from target data. This is a **test-time statistics adaptation** step rather than supervised target-label fine-tuning. fileciteturn1file0L442-L512

In [41]:
# ============================================================
# CELL 08 — SupCon loss and AdaBN adaptation
# ============================================================

# -----------------------------
# 3. LOSSES
# -----------------------------
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]

        sim = torch.matmul(features, features.T) / self.temperature

        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        logits_mask = torch.ones_like(mask)
        logits_mask.fill_diagonal_(0)
        mask = mask * logits_mask

        exp_logits = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)

        positives = mask.sum(1)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (positives + 1e-6)

        valid = positives > 0
        if valid.any():
            return -mean_log_prob_pos[valid].mean()
        return torch.zeros((), device=device, requires_grad=True)


def apply_adabn(model, target_dataloader, device, adaptation_trials=20):
    model.eval()

    target_samples = []
    trials_count = 0

    for x, _, _ in target_dataloader:
        target_samples.append(x)
        trials_count += x.size(0)
        if trials_count >= adaptation_trials:
            break

    if not target_samples:
        return model

    target_x = torch.cat(target_samples, dim=0)[:adaptation_trials].to(device)

    bn_modules = [
        m for m in model.modules()
        if isinstance(m, nn.modules.batchnorm._BatchNorm)
    ]

    if not bn_modules:
        return model

    saved = []
    for module in bn_modules:
        saved.append((module.training, module.momentum))
        module.reset_running_stats()
        module.momentum = 1.0
        module.train()

    with torch.no_grad():
        _ = model(target_x, lambda_grl=0.0)

    for module, (was_training, old_momentum) in zip(bn_modules, saved):
        module.momentum = 0.1
        module.eval()

    model.eval()
    return model


## CELL 09 — Model smoke test

Before launching the expensive experiment, this cell creates a random tensor with the expected EEG size and verifies that the complete model executes.

Expected:
- input `(2, 22, 1000)`
- 4 class logits
- 109 subject/domain logits
- 128-dimensional contrastive projection

It also reports the parameter count. fileciteturn1file0L515-L532

In [42]:
# ============================================================
# CELL 09 — Model smoke test
# ============================================================

# Run this cell only AFTER Cells 01–08 have been executed.
# This guard turns a confusing NameError into a direct instruction.
required = [
    "torch", "nn", "F", "N_CLASSES", "DOMAIN_CLASSES", "DEVICE",
    "N_CHANNELS", "FS", "TMAX", "S3MambaDA"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing notebook definitions: " + ", ".join(missing) +
        ". Run Cells 01 through 08 in order before running Cell 09."
    )


def smoke_test():
    model = S3MambaDA(
        num_classes=N_CLASSES,
        num_subjects=DOMAIN_CLASSES
    ).to(DEVICE)

    x = torch.randn(
        2,
        N_CHANNELS,
        int(FS * TMAX),
        device=DEVICE
    )

    with torch.no_grad():
        class_logits, domain_logits, z_proj = model(
            x,
            lambda_grl=0.0
        )

    print("Smoke test PASSED")
    print("  input:          ", tuple(x.shape))
    print("  class logits:   ", tuple(class_logits.shape))
    print("  domain logits:  ", tuple(domain_logits.shape))
    print("  projection:     ", tuple(z_proj.shape))
    print("  parameters:     ", sum(p.numel() for p in model.parameters()))


smoke_test()


Smoke test PASSED
  input:           (2, 22, 1000)
  class logits:    (2, 2)
  domain logits:   (2, 109)
  projection:      (2, 128)
  parameters:      62751


## CELL 10 — Training and subject-independent evaluation

`evaluate_large_scale_loso()` is the main experiment.

For each selected held-out subject:
1. choose the test subject;
2. sample up to 99 other available subjects for training;
3. load all selected runs for those subjects;
4. train a fresh model for 100 epochs;
5. ramp the GRL strength from approximately 0 toward 1 using a logistic schedule;
6. optimize classification + domain + contrastive losses;
7. apply AdaBN using all available target trials;
8. evaluate the target subject without labels during adaptation;
9. collect predictions, probabilities, embeddings, and per-epoch losses.

**Important terminology:** the function is named `evaluate_large_scale_loso`, but with 109 subjects it samples 10 test subjects, and each fold uses 99 of the other 108 subjects for training. The remaining 9 subjects are not used in that fold. Therefore this implementation is better described as a **10-fold sampled subject-held-out evaluation** rather than complete 109-fold LOSO. fileciteturn1file0L538-L581

In [43]:
# ============================================================
# CELL 10 — EXACT 10-SUBJECT BINARY EVALUATION
# ============================================================

def run_exact_binary_experiment(
    data_dir=DATA_DIR,
    test_subjects=TEST_SUBJECTS,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    seed=SEED
):

    seed_everything(seed)

    available = discover_available_subjects(
        data_dir,
        TOTAL_SUBJECTS
    )

    print(
        f"Available subjects: {len(available)}"
    )

    # --------------------------------------------------------
    # Verify requested subjects
    # --------------------------------------------------------

    missing = [
        s
        for s in test_subjects
        if s not in available
    ]

    if missing:

        raise RuntimeError(
            "Requested test subjects are missing: "
            +
            ", ".join(
                f"S{s:03d}"
                for s in missing
            )
        )

    # --------------------------------------------------------
    # Results
    # --------------------------------------------------------

    fold_rows = []

    pred_rows = []

    epoch_rows = []

    embedding_rows = []

    # --------------------------------------------------------
    # EXACT requested subjects
    # --------------------------------------------------------

    for fold_idx, test_subject in enumerate(
        test_subjects,
        start=1
    ):

        print()
        print("=" * 80)
        print(
            f"FOLD {fold_idx}/{len(test_subjects)} "
            f"| TEST SUBJECT S{test_subject:03d}"
        )
        print("=" * 80)

        # ----------------------------------------------------
        # Source subjects
        # ----------------------------------------------------

        train_subjects = [
            s
            for s in available
            if s != test_subject
        ]

        print(
            f"Source subjects: {len(train_subjects)}"
        )

        # ----------------------------------------------------
        # Load source data
        # ----------------------------------------------------

        train_dataset = EEGMMIDB_Dataset(
            data_dir,
            subjects=train_subjects
        )

        # ----------------------------------------------------
        # Load target data
        # ----------------------------------------------------

        test_dataset = EEGMMIDB_Dataset(
            data_dir,
            subjects=[test_subject]
        )

        print(
            f"Training trials: {len(train_dataset)}"
        )

        print(
            f"Testing trials : {len(test_dataset)}"
        )

        # ----------------------------------------------------
        # Verify exactly two classes
        # ----------------------------------------------------

        train_labels = np.asarray(
            train_dataset.labels
        )

        test_labels = np.asarray(
            test_dataset.labels
        )

        train_unique = np.unique(
            train_labels
        )

        test_unique = np.unique(
            test_labels
        )

        print(
            "Train classes:",
            train_unique.tolist()
        )

        print(
            "Test classes :",
            test_unique.tolist()
        )

        if len(test_unique) < 2:

            raise RuntimeError(
                f"S{test_subject:03d} does not contain "
                f"both binary classes. "
                f"Found: {test_unique.tolist()}"
            )

        # ----------------------------------------------------
        # DataLoaders
        # ----------------------------------------------------

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            drop_last=True
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=0
        )

        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        model = S3MambaDA(
            num_classes=2,
            num_subjects=DOMAIN_CLASSES
        ).to(DEVICE)

        # ----------------------------------------------------
        # Losses
        # ----------------------------------------------------

        criterion_cls = nn.CrossEntropyLoss(
            label_smoothing=LABEL_SMOOTHING
        )

        criterion_domain = nn.CrossEntropyLoss()

        criterion_supcon = SupConLoss(
            temperature=SUPCON_TEMP
        )

        # ----------------------------------------------------
        # Optimizer
        # ----------------------------------------------------

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )

        # ----------------------------------------------------
        # Cosine schedule
        # ----------------------------------------------------

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=num_epochs,
            eta_min=MIN_LR
        )

        # ----------------------------------------------------
        # AMP
        # ----------------------------------------------------

        use_amp = (
            DEVICE.type == "cuda"
        )

        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=use_amp
        )

        # ----------------------------------------------------
        # Best model
        # ----------------------------------------------------

        best_loss = float("inf")

        best_state = None

        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        for epoch in range(
            num_epochs
        ):

            model.train()

            total_loss_sum = 0.0
            cls_loss_sum = 0.0
            domain_loss_sum = 0.0
            supcon_loss_sum = 0.0

            total_samples = 0

            total_batches = len(
                train_loader
            )

            for batch_idx, (
                x,
                y,
                subject
            ) in enumerate(
                train_loader
            ):

                x = x.to(DEVICE)
                y = y.to(DEVICE)
                subject = subject.to(DEVICE)

                # ------------------------------------------------
                # GRL schedule
                # ------------------------------------------------

                progress = (
                    epoch * total_batches
                    +
                    batch_idx
                ) / max(
                    1,
                    num_epochs * total_batches
                )

                lambda_grl = (
                    2.0
                    /
                    (
                        1.0
                        +
                        np.exp(
                            -10.0 * progress
                        )
                    )
                    -
                    1.0
                )

                optimizer.zero_grad(
                    set_to_none=True
                )

                with torch.autocast(
                    device_type=DEVICE.type,
                    enabled=use_amp
                ):

                    (
                        logits,
                        domain_logits,
                        z_proj
                    ) = model(
                        x,
                        lambda_grl=lambda_grl
                    )

                    # --------------------------------------------
                    # Binary task
                    # --------------------------------------------

                    loss_cls = criterion_cls(
                        logits,
                        y
                    )

                    # --------------------------------------------
                    # Subject domain
                    # --------------------------------------------

                    loss_domain = criterion_domain(
                        domain_logits,
                        subject
                    )

                    # --------------------------------------------
                    # SupCon
                    # --------------------------------------------

                    loss_supcon = criterion_supcon(
                        z_proj,
                        y
                    )

                    # --------------------------------------------
                    # Total
                    # --------------------------------------------

                    loss = (
                        loss_cls
                        +
                        DOMAIN_WEIGHT * loss_domain
                        +
                        SUPCON_WEIGHT * loss_supcon
                    )

                # ------------------------------------------------
                # Backprop
                # ------------------------------------------------

                scaler.scale(
                    loss
                ).backward()

                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    GRAD_CLIP
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                # ------------------------------------------------
                # Statistics
                # ------------------------------------------------

                bs = x.size(0)

                total_samples += bs

                total_loss_sum += (
                    loss.item() * bs
                )

                cls_loss_sum += (
                    loss_cls.item() * bs
                )

                domain_loss_sum += (
                    loss_domain.item() * bs
                )

                supcon_loss_sum += (
                    loss_supcon.item() * bs
                )

            scheduler.step()

            avg_loss = (
                total_loss_sum
                /
                max(
                    1,
                    total_samples
                )
            )

            avg_cls = (
                cls_loss_sum
                /
                max(
                    1,
                    total_samples
                )
            )

            avg_domain = (
                domain_loss_sum
                /
                max(
                    1,
                    total_samples
                )
            )

            avg_supcon = (
                supcon_loss_sum
                /
                max(
                    1,
                    total_samples
                )
            )

            # ----------------------------------------------------
            # History
            # ----------------------------------------------------

            epoch_rows.append({

                "fold":
                    fold_idx,

                "test_subject":
                    test_subject,

                "epoch":
                    epoch + 1,

                "loss":
                    avg_loss,

                "classification_loss":
                    avg_cls,

                "domain_loss":
                    avg_domain,

                "supcon_loss":
                    avg_supcon,

                "lambda_grl":
                    lambda_grl,

                "learning_rate":
                    optimizer.param_groups[0]["lr"]
            })

            # ----------------------------------------------------
            # Save best model
            # ----------------------------------------------------

            if avg_loss < best_loss:

                best_loss = avg_loss

                best_state = {
                    k:
                    v.detach()
                    .cpu()
                    .clone()

                    for k, v
                    in model.state_dict().items()
                }

            # ----------------------------------------------------
            # Progress
            # ----------------------------------------------------

            if (
                epoch == 0
                or
                (epoch + 1) % 10 == 0
            ):

                print(
                    f"Epoch {epoch+1:03d}/{num_epochs} | "
                    f"Total={avg_loss:.4f} | "
                    f"Cls={avg_cls:.4f} | "
                    f"Domain={avg_domain:.4f} | "
                    f"SupCon={avg_supcon:.4f} | "
                    f"LR={optimizer.param_groups[0]['lr']:.2e}"
                )

        # --------------------------------------------------------
        # Restore best training checkpoint
        # --------------------------------------------------------

        if best_state is not None:

            model.load_state_dict(
                best_state
            )

        # --------------------------------------------------------
        # AdaBN
        # --------------------------------------------------------

        model = apply_adabn(
            model,
            test_loader,
            DEVICE,
            adaptation_trials=len(
                test_dataset
            )
        )

        model.eval()

        all_labels = []
        all_predictions = []
        all_probabilities = []
        all_embeddings = []

        # --------------------------------------------------------
        # Test
        # --------------------------------------------------------

        with torch.no_grad():

            for x, y, subject in test_loader:

                x = x.to(DEVICE)

                (
                    logits,
                    _,
                    z_proj
                ) = model(
                    x,
                    lambda_grl=0.0
                )

                probabilities = torch.softmax(
                    logits,
                    dim=1
                )

                predictions = torch.argmax(
                    probabilities,
                    dim=1
                )

                all_labels.extend(
                    y.numpy().tolist()
                )

                all_predictions.extend(
                    predictions.cpu()
                    .numpy()
                    .tolist()
                )

                all_probabilities.append(
                    probabilities.cpu()
                    .numpy()
                )

                all_embeddings.append(
                    z_proj.cpu()
                    .numpy()
                )

        probabilities = np.concatenate(
            all_probabilities,
            axis=0
        )

        embeddings = np.concatenate(
            all_embeddings,
            axis=0
        )

        # --------------------------------------------------------
        # Metrics
        # --------------------------------------------------------

        accuracy = accuracy_score(
            all_labels,
            all_predictions
        )

        balanced_accuracy = (
            balanced_accuracy_score(
                all_labels,
                all_predictions
            )
        )

        kappa = cohen_kappa_score(
            all_labels,
            all_predictions
        )

        precision = precision_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )

        recall = recall_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )

        f1 = f1_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        )

        # --------------------------------------------------------
        # AUC
        # --------------------------------------------------------

        try:

            roc_auc = roc_auc_score(
                all_labels,
                probabilities[:, 1]
            )

        except Exception:

            roc_auc = np.nan

        # --------------------------------------------------------
        # Confusion matrix
        # --------------------------------------------------------

        cm = confusion_matrix(
            all_labels,
            all_predictions,
            labels=[0, 1]
        )

        print()
        print(
            f"Subject S{test_subject:03d}"
        )

        print(
            f"  Accuracy       : {accuracy*100:.2f}%"
        )

        print(
            f"  Balanced Acc   : {balanced_accuracy*100:.2f}%"
        )

        print(
            f"  Kappa          : {kappa:.4f}"
        )

        print(
            f"  ROC-AUC        : {roc_auc:.4f}"
        )

        print(
            f"  Macro Precision: {precision:.4f}"
        )

        print(
            f"  Macro Recall   : {recall:.4f}"
        )

        print(
            f"  Macro F1       : {f1:.4f}"
        )

        print()
        print("Confusion matrix:")
        print(cm)

        # --------------------------------------------------------
        # Store fold
        # --------------------------------------------------------

        fold_rows.append({

            "fold":
                fold_idx,

            "test_subject":
                test_subject,

            "n_train_trials":
                len(train_dataset),

            "n_test_trials":
                len(test_dataset),

            "accuracy":
                accuracy,

            "balanced_accuracy":
                balanced_accuracy,

            "kappa":
                kappa,

            "roc_auc":
                roc_auc,

            "precision_macro":
                precision,

            "recall_macro":
                recall,

            "f1_macro":
                f1
        })

        # --------------------------------------------------------
        # Store predictions
        # --------------------------------------------------------

        for i in range(
            len(all_labels)
        ):

            pred_rows.append({

                "fold":
                    fold_idx,

                "test_subject":
                    test_subject,

                "true_label":
                    int(all_labels[i]),

                "pred_label":
                    int(all_predictions[i]),

                "prob_left":
                    float(
                        probabilities[i, 0]
                    ),

                "prob_right":
                    float(
                        probabilities[i, 1]
                    )
            })

            embedding_rows.append({

                "fold":
                    fold_idx,

                "test_subject":
                    test_subject,

                "true_label":
                    int(all_labels[i]),

                **{
                    f"z_{j}":
                    float(
                        embeddings[i, j]
                    )

                    for j in range(
                        embeddings.shape[1]
                    )
                }
            })

        # --------------------------------------------------------
        # Cleanup
        # --------------------------------------------------------

        del model

        del train_loader
        del test_loader

        del train_dataset
        del test_dataset

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ============================================================
    # DATAFRAMES
    # ============================================================

    fold_df = pd.DataFrame(
        fold_rows
    )

    pred_df = pd.DataFrame(
        pred_rows
    )

    epoch_df = pd.DataFrame(
        epoch_rows
    )

    emb_df = pd.DataFrame(
        embedding_rows
    )

    # ============================================================
    # SAVE
    # ============================================================

    fold_df.to_csv(
        RESULTS_DIR /
        "fold_metrics.csv",
        index=False
    )

    pred_df.to_csv(
        RESULTS_DIR /
        "test_predictions.csv",
        index=False
    )

    epoch_df.to_csv(
        RESULTS_DIR /
        "epoch_history.csv",
        index=False
    )

    emb_df.to_csv(
        RESULTS_DIR /
        "test_embeddings.csv",
        index=False
    )

    # ============================================================
    # FINAL RESULTS
    # ============================================================

    mean_accuracy = (
        fold_df["accuracy"].mean()
    )

    std_accuracy = (
        fold_df["accuracy"].std(
            ddof=0
        )
    )

    mean_balanced_accuracy = (
        fold_df[
            "balanced_accuracy"
        ].mean()
    )

    mean_kappa = (
        fold_df["kappa"].mean()
    )

    mean_auc = (
        fold_df["roc_auc"].mean()
    )

    pooled_accuracy = accuracy_score(
        pred_df["true_label"],
        pred_df["pred_label"]
    )

    print()
    print("=" * 80)
    print(
        "BINARY LEFT-vs-RIGHT RESULTS"
    )
    print("=" * 80)

    print(
        f"Completed folds: {len(fold_df)}"
    )

    print(
        "Test subjects:",
        ", ".join(
            f"S{s:03d}"
            for s in test_subjects
        )
    )

    print()
    print(
        f"Pooled Accuracy: "
        f"{pooled_accuracy*100:.2f}%"
    )

    print(
        f"Mean Fold Accuracy: "
        f"{mean_accuracy*100:.2f}% "
        f"± {std_accuracy*100:.2f}%"
    )

    print(
        f"Mean Balanced Accuracy: "
        f"{mean_balanced_accuracy*100:.2f}%"
    )

    print(
        f"Mean Cohen's Kappa: "
        f"{mean_kappa:.4f}"
    )

    print(
        f"Mean ROC-AUC: "
        f"{mean_auc:.4f}"
    )

    print(
        "Binary chance level: 50.00%"
    )

    print()
    print("Per-fold results:")
    print(
        fold_df[
            [
                "test_subject",
                "accuracy",
                "balanced_accuracy",
                "kappa",
                "roc_auc"
            ]
        ].to_string(
            index=False
        )
    )

    return (
        fold_df,
        pred_df,
        epoch_df,
        emb_df
    )

## CELL 11 — Architecture and training-loss figures

These functions create paper-style figures from the experiment outputs.

`plot_architecture()` draws the conceptual network.
`plot_training_curves()` aggregates losses across folds by epoch and plots total, classification, domain, and SupCon losses. fileciteturn1file0L808-L894

In [44]:
# ============================================================
# CELL 11 — Architecture and training-loss plots
# ============================================================

# ------------------------------------------------------------
# 6. PAPER-READY FIGURES / INFOGRAPHICS
# ------------------------------------------------------------

def plot_architecture():
    fig, ax = plt.subplots(figsize=(15, 7))
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 8)
    ax.axis("off")

    blocks = [
        (0.3, 5.5, 1.7, 1.0, "Raw EEG\n(B,22,1000)"),
        (2.4, 5.5, 1.9, 1.0, "Per-trial\nZ-score"),
        (4.7, 5.5, 2.0, 1.0, "Learnable\nSinc Bank\n10 bands"),
        (7.1, 5.5, 2.0, 1.0, "Dynamic GNN\nAdaptive graph\n22 → 64"),
        (9.5, 5.5, 2.0, 1.0, "BiGRU\nTemporal\nmodeling"),
        (11.9, 5.5, 2.2, 1.0, "SE Attention\n+ mean pool\nz ∈ R64"),
    ]

    for x, y, w, h, txt in blocks:
        r = plt.Rectangle((x, y), w, h, fill=False, linewidth=1.8)
        ax.add_patch(r)
        ax.text(x+w/2, y+h/2, txt, ha="center", va="center", fontsize=11)

    for i in range(len(blocks)-1):
        x1 = blocks[i][0] + blocks[i][2]
        x2 = blocks[i+1][0]
        y = blocks[i][1] + blocks[i][3]/2
        ax.annotate("", xy=(x2, y), xytext=(x1, y),
                    arrowprops=dict(arrowstyle="->", linewidth=1.5))

    # Branches
    ax.plot([13.0, 13.0], [5.5, 3.8], linewidth=1.5)
    ax.annotate("", xy=(10.7, 3.2), xytext=(13.0, 3.8),
                arrowprops=dict(arrowstyle="->", linewidth=1.5))
    ax.annotate("", xy=(13.0, 1.8), xytext=(13.0, 3.8),
                arrowprops=dict(arrowstyle="->", linewidth=1.5))
    ax.annotate("", xy=(6.8, 1.8), xytext=(13.0, 3.8),
                arrowprops=dict(arrowstyle="->", linewidth=1.5))

    outputs = [
        (5.4, 0.7, 2.8, 1.0, "Class Head\n2 MI classes"),
        (9.1, 0.7, 3.2, 1.0, "GRL + Domain Head\nSubject alignment"),
        (3.2, 2.6, 3.6, 1.0, "Projection Head\n128-D SupCon embedding"),
    ]

    for x, y, w, h, txt in outputs:
        r = plt.Rectangle((x, y), w, h, fill=False, linewidth=1.6)
        ax.add_patch(r)
        ax.text(x+w/2, y+h/2, txt, ha="center", va="center", fontsize=10)

    ax.text(7.5, 7.6,
            "S³ Binary Spectral-Spatial Domain-Adaptive EEG Classification Pipeline",
            ha="center", va="center", fontsize=16, fontweight="bold")
    ax.text(7.5, 7.1,
            "Code-aligned architecture: learnable frequency decomposition + dynamic topology + bidirectional GRU + GRL + SupCon + AdaBN",
            ha="center", va="center", fontsize=10)

    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig1_Architecture.png", dpi=400, bbox_inches="tight")
    plt.close(fig)

def plot_training_curves(epoch_df):
    if epoch_df.empty:
        return

    # Aggregate across folds by epoch.
    g = epoch_df.groupby("epoch").agg({
        "loss_total": "mean",
        "loss_cls": "mean",
        "loss_domain": "mean",
        "loss_supcon": "mean",
    }).reset_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(g["epoch"], g["loss_total"], label="Total loss", linewidth=2)
    ax.plot(g["epoch"], g["loss_cls"], label="Classification loss", linewidth=1.5)
    ax.plot(g["epoch"], g["loss_domain"], label="Domain loss", linewidth=1.5)
    ax.plot(g["epoch"], g["loss_supcon"], label="SupCon loss", linewidth=1.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Training Loss Components")
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig2_TrainingLoss.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


## CELL 12 — Performance figures

These functions visualize:
- held-out-subject accuracy;
- normalized confusion matrix;
- precision, recall, and F1 for the four classes;
- one-vs-rest ROC curves and AUC.

They read the CSV artifacts generated by the evaluation function. fileciteturn1file0L897-L1007

In [45]:
# ============================================================
# CELL 12 — Binary performance figures
# ============================================================

def plot_subject_accuracy(fold_df):
    if fold_df.empty:
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    labels = [f"S{s:03d}" for s in fold_df["test_subject"]]
    ax.bar(labels, fold_df["accuracy"] * 100)
    ax.axhline(50, linestyle="--", linewidth=1.3, label="Binary chance (50%)")
    ax.set_ylabel("Accuracy (%)")
    ax.set_xlabel("Held-out subject")
    ax.set_ylim(0, 100)
    ax.set_title("Binary Subject-Independent Accuracy by Fold")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig3_BinaryFoldAccuracy.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


def plot_confusion(pred_df):
    if pred_df.empty:
        return

    cm = confusion_matrix(
        pred_df["true_label"],
        pred_df["pred_label"],
        labels=list(range(N_CLASSES))
    )

    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, interpolation="nearest")
    ax.set_title("Normalized Binary Confusion Matrix")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_xticks(range(N_CLASSES), CLASS_NAMES, rotation=20, ha="right")
    ax.set_yticks(range(N_CLASSES), CLASS_NAMES)

    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(
                j, i,
                f"{cm_norm[i, j]*100:.1f}%",
                ha="center", va="center"
            )

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig4_BinaryConfusionMatrix.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


def plot_class_metrics(pred_df):
    if pred_df.empty:
        return

    report = classification_report(
        pred_df["true_label"],
        pred_df["pred_label"],
        labels=list(range(N_CLASSES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    precision = [report[c]["precision"] * 100 for c in CLASS_NAMES]
    recall = [report[c]["recall"] * 100 for c in CLASS_NAMES]
    f1 = [report[c]["f1-score"] * 100 for c in CLASS_NAMES]

    x = np.arange(N_CLASSES)
    width = 0.25

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x-width, precision, width, label="Precision")
    ax.bar(x, recall, width, label="Recall")
    ax.bar(x+width, f1, width, label="F1")
    ax.set_xticks(x, CLASS_NAMES, rotation=20, ha="right")
    ax.set_ylim(0, 100)
    ax.set_ylabel("Score (%)")
    ax.set_title("Binary Per-Class Metrics")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig5_BinaryClassMetrics.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


def plot_roc(pred_df):
    if pred_df.empty:
        return

    y_true = pred_df["true_label"].to_numpy()
    positive_prob = pred_df["prob_1"].to_numpy()

    fig, ax = plt.subplots(figsize=(7, 6))

    if np.unique(y_true).size == 2:
        fpr, tpr, _ = roc_curve(y_true, positive_prob)
        roc_auc = auc(fpr, tpr)
        ax.plot(
            fpr, tpr, linewidth=2,
            label=f"Right Hand MI (AUC={roc_auc:.3f})"
        )

    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Binary ROC Curve")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig6_BinaryROC_AUC.png", dpi=400, bbox_inches="tight")
    plt.close(fig)


## CELL 13 — Embedding visualization and figure generation

`plot_tsne()` reduces the saved 128-D contrastive embeddings to 2-D using t-SNE and colors points by the true EEG class.

`generate_all_figures()` first generates the architecture figure, then checks whether all result CSV files exist. If they do, it generates the remaining six figures and saves them under `results_s3_da/figures/`. fileciteturn2file0L10-L75

In [46]:
# ============================================================
# CELL 13 — t-SNE and figure generation
# ============================================================

def plot_tsne(emb_df, seed=SEED):
    if emb_df.empty:
        return

    z_cols = [c for c in emb_df.columns if c.startswith("z_")]
    if len(emb_df) < 10 or len(z_cols) < 2:
        return

    X = emb_df[z_cols].to_numpy()
    y = emb_df["true_label"].to_numpy()

    perplexity = min(30, max(5, (len(X) - 1) // 3))

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=seed
    )

    Z = tsne.fit_transform(X)

    fig, ax = plt.subplots(figsize=(8, 6))
    for c in range(N_CLASSES):
        mask = y == c
        ax.scatter(Z[mask, 0], Z[mask, 1], s=12, label=CLASS_NAMES[c], alpha=0.75)

    ax.set_title("t-SNE Projection of Binary Test-Time Class Embeddings")
    ax.set_xlabel("t-SNE dimension 1")
    ax.set_ylabel("t-SNE dimension 2")
    ax.legend(markerscale=1.5, fontsize=8)
    ax.grid(True, alpha=0.15)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig7_tSNE.png", dpi=400, bbox_inches="tight")
    plt.close(fig)

def generate_all_figures():
    plot_architecture()

    required = [
        "fold_metrics.csv",
        "test_predictions.csv",
        "epoch_history.csv",
        "test_embeddings.csv"
    ]

    if not all((RESULTS_DIR / f).exists() for f in required):
        print("Only architecture figure was generated. Run the experiment first.")
        return

    fold_df = pd.read_csv(RESULTS_DIR / "fold_metrics.csv")
    pred_df = pd.read_csv(RESULTS_DIR / "test_predictions.csv")
    epoch_df = pd.read_csv(RESULTS_DIR / "epoch_history.csv")
    emb_df = pd.read_csv(RESULTS_DIR / "test_embeddings.csv")

    plot_training_curves(epoch_df)
    plot_subject_accuracy(fold_df)
    plot_confusion(pred_df)
    plot_class_metrics(pred_df)
    plot_roc(pred_df)
    plot_tsne(emb_df)

    print("Figures written to:", FIG_DIR.resolve())
    for p in sorted(FIG_DIR.glob("*.png")):
        print(" -", p.name)


## CELL 14 — Run the experiment

Run this cell only after the smoke test succeeds.

For a cheap pipeline check, temporarily set `NUM_EPOCHS` to a small value such as 1–2 in CELL 02. For the full source configuration, restore it to 100.

The original notebook calls figure generation immediately, but the actual training call is left commented out. This cell makes the intended execution order explicit.

In [47]:
# ============================================================
# CELL 14 — Run the binary experiment
# ============================================================

# ------------------------------------------------------------
# FAST SANITY TEST
# ------------------------------------------------------------
# Use this first if you are verifying the pipeline.
# Uncomment these two lines, execute this cell, then restore the
# original configuration above for the final experiment.
# NUM_EPOCHS = 2
# NUM_TEST_FOLDS = 1

print("Starting binary subject-independent experiment...")
print("Task:", "Left Hand MI vs Right Hand MI")
print("Chance level: 50.00%")

fold_df, pred_df, epoch_df, emb_df = evaluate_binary_subject_independent()
generate_all_figures()


Starting binary subject-independent experiment...
Task: Left Hand MI vs Right Hand MI
Chance level: 50.00%
Available subject folders: 109
FOLD 1/10 | TEST SUBJECT S082
[OK] S096 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S096 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S096 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S070 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S070 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S070 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S012 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S012 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S012 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S076 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S076 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S076 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S055 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S055 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S055 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S005 R04 | T1=2->Left | T2=3->Ri

KeyboardInterrupt: 

## CELL 15 — Export paper-ready results text

This helper reads `fold_metrics.csv` and produces a human-readable `paper_results.txt` containing:
- number of completed folds;
- held-out subject IDs;
- mean ± standard deviation of accuracy;
- mean ± standard deviation of Cohen's kappa;
- the full per-fold metrics table.

In [34]:
# ============================================================
# CELL 15 — Export binary paper-ready results text
# ============================================================

def export_results_text():
    fold_file = RESULTS_DIR / "fold_metrics.csv"

    if not fold_file.exists():
        print("Run the binary experiment first.")
        return

    fold_df = pd.read_csv(fold_file)
    pred_file = RESULTS_DIR / "test_predictions.csv"
    pred_df = pd.read_csv(pred_file) if pred_file.exists() else pd.DataFrame()

    if fold_df.empty:
        print("No completed folds.")
        return

    acc_mean = fold_df["accuracy"].mean() * 100
    acc_std = fold_df["accuracy"].std(ddof=0) * 100
    bal_mean = fold_df["balanced_accuracy"].mean() * 100
    kappa_mean = fold_df["kappa"].mean()
    auc_mean = fold_df["roc_auc"].mean()

    pooled_acc = (
        accuracy_score(pred_df["true_label"], pred_df["pred_label"]) * 100
        if len(pred_df) else np.nan
    )

    text = f"""
BINARY LEFT-vs-RIGHT MOTOR-IMAGERY RESULTS
==========================================
Completed folds: {len(fold_df)}
Test subjects: {", ".join("S%03d" % s for s in fold_df["test_subject"])}

Pooled Accuracy: {pooled_acc:.2f}%
Mean Fold Accuracy: {acc_mean:.2f}% ± {acc_std:.2f}%
Mean Balanced Accuracy: {bal_mean:.2f}%
Mean Cohen's Kappa: {kappa_mean:.4f}
Mean ROC-AUC: {auc_mean:.4f}
Binary chance level: 50.00%

Per-fold results:
{fold_df.to_string(index=False)}
""".strip()

    out_file = RESULTS_DIR / "binary_paper_results.txt"
    out_file.write_text(text)
    print(text)
    print("\nSaved:", out_file.resolve())

export_results_text()


BINARY LEFT-vs-RIGHT MOTOR-IMAGERY RESULTS
Completed folds: 10
Test subjects: S082, S015, S004, S095, S036, S032, S029, S018, S014, S087

Pooled Accuracy: 70.67%
Mean Fold Accuracy: 70.67% ± 10.41%
Mean Balanced Accuracy: 70.70%
Mean Cohen's Kappa: 0.4136
Mean ROC-AUC: 0.7448
Binary chance level: 50.00%

Per-fold results:
 fold  test_subject  n_train_trials  n_test_trials  accuracy  balanced_accuracy    kappa  roc_auc  precision_macro  recall_macro  f1_macro
    1            82            4467             45  0.800000           0.800395 0.600197 0.729249         0.800395      0.800395  0.800000
    2            15            4467             45  0.777778           0.777668 0.555336 0.867589         0.777668      0.777668  0.777668
    3             4            4467             45  0.733333           0.733202 0.466403 0.822134         0.733202      0.733202  0.733202
    4            95            4467             45  0.600000           0.600791 0.201183 0.636364         0.601190      

## CELL 16 — Execution order and expected outputs

Run the notebook in order after restarting the kernel:

**01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09 → 10 → 11 → 12 → 13 → 14 → 15**

The final binary experiment uses:

- **Runs:** 4, 8, 12
- **Class 0:** Left Hand MI
- **Class 1:** Right Hand MI
- **Classes:** 2
- **Chance accuracy:** 50%
- **Test folds:** 10
- **Training subjects per fold:** up to 99

Saved results:

`results_s3_binary_da/fold_metrics.csv`

`results_s3_binary_da/test_predictions.csv`

`results_s3_binary_da/epoch_history.csv`

`results_s3_binary_da/test_embeddings.csv`

`results_s3_binary_da/summary.json`

Figures include binary fold accuracy, normalized confusion matrix, precision/recall/F1, ROC-AUC, training losses, architecture, and t-SNE.
